In [1]:
!pip install langchain langchain-groq langsmith -q

In [2]:
import os
import json

os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"
os.environ["LANGCHAIN_API_KEY"] = "your_langsmith_api_key_here"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Resume-Screening-Task3"

print("Environment configured.")

Environment configured.


In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=1024
)

print("LLM ready:", llm.model_name)

LLM ready: llama-3.3-70b-versatile


In [4]:
from langchain_core.prompts import PromptTemplate

# Step 1: Skill Extraction
extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
You are a resume parser. Extract structured information from the resume below.

Return ONLY a valid JSON object in this exact format:
{{
  "skills": ["skill1", "skill2"],
  "tools": ["tool1", "tool2"],
  "experience": "X years / Fresher / etc."
}}

Rules:
- Do NOT assume or invent anything not written in the resume
- Only include what is explicitly mentioned

Resume:
{resume}

JSON:
"""
)

# Step 2: Matching
match_prompt = PromptTemplate(
    input_variables=["resume_data", "job_description"],
    template="""
You are a technical recruiter. Compare the candidate's profile against the job description.

Return ONLY a valid JSON object:
{{
  "matched_skills": ["skill1", "skill2"],
  "missing_skills": ["skill3", "skill4"]
}}

Rules:
- Only include skills explicitly in both the resume and JD for matched
- Missing skills must be in the JD but absent from the resume

Candidate Profile:
{resume_data}

Job Description:
{job_description}

JSON:
"""
)

# Step 3: Scoring
score_prompt = PromptTemplate(
    input_variables=["matched_skills", "missing_skills"],
    template="""
You are a hiring evaluator. Assign a fit score from 0 to 100 for this candidate.

Return ONLY a valid JSON object:
{{
  "score": <number between 0 and 100>
}}

Scoring logic:
- More matched skills = higher score
- More missing critical skills = lower score
- Be realistic and consistent

Matched Skills: {matched_skills}
Missing Skills: {missing_skills}

JSON:
"""
)

# Step 4: Explanation
explain_prompt = PromptTemplate(
    input_variables=["score", "matched_skills", "missing_skills"],
    template="""
You are a recruitment assistant. Write a clear, concise explanation for why a candidate received this score.

Score: {score}/100
Matched Skills: {matched_skills}
Missing Skills: {missing_skills}

Write 3-4 sentences explaining:
- What the candidate did well
- What they are lacking
- Whether they should be shortlisted

Explanation:
"""
)

print("Prompts defined.")

Prompts defined.


In [5]:
# Build individual chains using LCEL pipe syntax
extract_chain = extract_prompt | llm
match_chain   = match_prompt   | llm
score_chain   = score_prompt   | llm
explain_chain = explain_prompt | llm

print("Chains built.")

Chains built.


In [6]:
def parse_json(response_text):
    """Safely parse JSON from LLM response string."""
    try:
        # Strip markdown code fences if present
        text = response_text.strip()
        if "```" in text:
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text.strip())
    except Exception as e:
        print(f"JSON parse warning: {e}")
        return {}

def run_pipeline(candidate_name, resume, job_description):
    print(f"\n{'='*50}")
    print(f"Processing: {candidate_name}")
    print(f"{'='*50}")

    # Step 1: Extract
    raw_extract = extract_chain.invoke({"resume": resume})
    extracted = parse_json(raw_extract.content)
    print(f"[Extracted] {extracted}")

    # Step 2: Match
    raw_match = match_chain.invoke({
        "resume_data": json.dumps(extracted),
        "job_description": job_description
    })
    matched = parse_json(raw_match.content)
    print(f"[Matched]   {matched}")

    # Step 3: Score
    raw_score = score_chain.invoke({
        "matched_skills": str(matched.get("matched_skills", [])),
        "missing_skills": str(matched.get("missing_skills", []))
    })
    scored = parse_json(raw_score.content)
    print(f"[Score]     {scored}")

    # Step 4: Explain
    raw_explain = explain_chain.invoke({
        "score": scored.get("score", "N/A"),
        "matched_skills": str(matched.get("matched_skills", [])),
        "missing_skills": str(matched.get("missing_skills", []))
    })
    explanation = raw_explain.content.strip()
    print(f"[Explanation]\n{explanation}")

    return {
        "candidate": candidate_name,
        "extracted": extracted,
        "matched": matched,
        "score": scored.get("score"),
        "explanation": explanation
    }

print("Pipeline function ready.")

Pipeline function ready.


In [7]:
job_description = """
We are hiring a Machine Learning Engineer.

Required Skills:
- Python
- Machine Learning (scikit-learn, XGBoost)
- Deep Learning (PyTorch or TensorFlow)
- MLOps (MLflow, Docker)
- REST API development (FastAPI or Flask)
- SQL and data preprocessing

Experience: 2+ years preferred.
"""

strong_resume = """
Name: Rahul Mehta
Experience: 3 years as ML Engineer at a fintech startup.

Skills:
- Python, Pandas, NumPy
- Machine Learning: scikit-learn, XGBoost, LightGBM
- Deep Learning: PyTorch, TensorFlow
- MLOps: MLflow, Docker, GitHub Actions
- REST APIs: FastAPI
- SQL, PostgreSQL

Projects:
- Built fraud detection model with 94% AUC using XGBoost
- Deployed PyTorch NLP model via FastAPI on AWS
"""

average_resume = """
Name: Sneha Reddy
Experience: 1 year internship at a data analytics firm.

Skills:
- Python, Pandas
- Machine Learning: scikit-learn (basic regression/classification)
- SQL

Projects:
- Customer churn prediction using Logistic Regression
- Data cleaning pipelines in Python
"""

weak_resume = """
Name: Arjun Pillai
Experience: Fresher, recent graduate.

Skills:
- MS Excel
- Basic Python (syntax only)
- PowerPoint

Projects:
- College project on sales data visualization using Excel charts
"""

print("Data ready.")

Data ready.


In [8]:
results = []

result_strong  = run_pipeline("Rahul Mehta (Strong)",  strong_resume,  job_description)
result_average = run_pipeline("Sneha Reddy (Average)", average_resume, job_description)
result_weak    = run_pipeline("Arjun Pillai (Weak)",   weak_resume,    job_description)

results = [result_strong, result_average, result_weak]


Processing: Rahul Mehta (Strong)
[Extracted] {'skills': ['Python', 'Pandas', 'NumPy', 'scikit-learn', 'XGBoost', 'LightGBM', 'PyTorch', 'TensorFlow', 'MLflow', 'Docker', 'GitHub Actions', 'FastAPI', 'SQL', 'PostgreSQL'], 'tools': ['MLflow', 'Docker', 'GitHub Actions', 'FastAPI'], 'experience': '3 years'}
[Matched]   {'matched_skills': ['Python', 'scikit-learn', 'XGBoost', 'PyTorch', 'TensorFlow', 'MLflow', 'Docker', 'FastAPI', 'SQL'], 'missing_skills': ['Flask']}
[Score]     {'score': 90}
[Explanation]
The candidate demonstrated exceptional proficiency in a wide range of skills, including Python, scikit-learn, XGBoost, PyTorch, TensorFlow, MLflow, Docker, and FastAPI, showcasing their strong foundation in machine learning and software development. However, they are lacking experience with Flask, a key framework required for the role. Despite this gap, the candidate's impressive array of matched skills earns them a score of 90/100, making them a strong contender for the position. Overa

In [9]:
print("\n" + "="*60)
print(f"{'CANDIDATE':<25} {'SCORE':>6}")
print("="*60)
for r in results:
    print(f"{r['candidate']:<25} {str(r['score']):>6}/100")
print("="*60)


CANDIDATE                  SCORE
Rahul Mehta (Strong)          90/100
Sneha Reddy (Average)         20/100
Arjun Pillai (Weak)            5/100


In [10]:
# Debugging cell: intentionally passing an empty resume to show how the pipeline handles bad input
# This demonstrates LangSmith tracing catching edge case failures

print("=== DEBUG RUN: Empty Resume ===")

bad_resume = ""  # intentionally empty

try:
    debug_result = run_pipeline("Debug Candidate (Empty Resume)", bad_resume, job_description)
    print("\n[Debug Result]")
    print(f"Score returned: {debug_result['score']}")
    print("Note: With an empty resume, the model should return no skills and a score near 0.")
    print("Check LangSmith trace to see how each step handled the empty input.")
except Exception as e:
    print(f"Pipeline error on bad input: {e}")
    print("This failure is visible in LangSmith trace for debugging.")

=== DEBUG RUN: Empty Resume ===

Processing: Debug Candidate (Empty Resume)
[Extracted] {}
[Matched]   {'matched_skills': [], 'missing_skills': ['Python', 'Machine Learning', 'Deep Learning', 'MLOps', 'REST API development', 'SQL and data preprocessing']}
[Score]     {'score': 0}
[Explanation]
Unfortunately, the candidate did not demonstrate any relevant skills for the position, leaving no notable strengths to highlight. They are lacking essential skills in key areas such as programming languages (Python), technical expertise (Machine Learning, Deep Learning), and development capabilities (MLOps, REST API development, SQL and data preprocessing). Given the significant gap in required skills, it is unlikely that this candidate would be a strong fit for the role. As a result, it is not recommended to shortlist this candidate for further consideration.

[Debug Result]
Score returned: 0
Note: With an empty resume, the model should return no skills and a score near 0.
Check LangSmith trace 